# AI News Developer Agent — Google ADK

## Getting started

This notebook shows how to scaffold, configure, and run a stateful, multi-agent AI news assistant using Google ADK and Gemini models. It guides you through project creation, tool integration, and assembling a RootAgent that routes requests to specialized agents.

This notebook demonstrates:

- Creating an ADK agent project with `adk create`.
- Implementing `agent.py` for a Python-based agent.
- Adding built-in and third-party tools (for example, `google_search`, `BuiltInCodeExecutor`).
- Designing a multi-agent architecture to orchestrate tools.
- Testing prompts and validating agent behavior.

In [1]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()
import os 

In [2]:
print("HF configured:", bool(os.getenv("HUGGING_FACE_TOKEN")))
print("GitHub configured:", bool(os.getenv("GITHUB_TOKEN")))


HF configured: True
GitHub configured: True


## Setting up the agent

Run the command below to scaffold the agent project and create the initial files and configuration. The generated folder contains a minimal Python package and an `.env` file for credentials.

In [ ]:
!adk create --type=code app_01 --model gemini-2.5-flash --api_key $GEMINI_API_KEY  

- The `adk create` command scaffolds a new project. The generated files include:

```bash
app_01/
    __init__.py    # marks the directory as a Python package
    agent.py       # main agent implementation
    .env           # stores API keys/config (do not commit)
```

Note: this example used `--type=code` to generate a Python-based agent.

- `--model` selects the LLM for the agent (for example, `gemini-2.5-flash`). Choose a model based on latency and text-processing needs.

- `--api_key` provides the Vertex AI (Gemini) API key required to call the model. See [Google Vertex AI documentation]() for creating and managing API keys.

**Workarounds for ADK tool restrictions**

ADK includes built-in tools (for example, `google_search` and code-execution tools) and integrations (for example, Hugging Face and GitHub). Some tools cannot be combined inside a single agent instance. To handle this, use the following pattern:

- Create specialized agents (for example, `SearchAgent`, `CodeAgent`).
- Expose each specialized agent via an `AgentTool` wrapper.
- Use a `RootAgent` to orchestrate and delegate requests to the appropriate specialized agent.

This separates responsibilities while respecting ADK constraints.

## Writing the `agent.py`

Use `adk create` to scaffold the project and then write files (for example with `%%writefile app_01/agent.py`).

### Adding tools to the agent

A standalone LLM is limited by its training cutoff and cannot fetch live information. To build an AI news assistant that retrieves current content, give the agent Tools. In ADK, “tool” has two meanings:

| Term                 | Meaning                                                                   |
| -------------------- | ------------------------------------------------------------------------- |
| **ADK Tool**         | A callable object exposed to an agent (for example, `google_search`, `AgentTool`) |
| **Third-Party Tool** | An external system or API (for example, Hugging Face, GitHub)             |

ADK provides powerful built-in tools (for example, `google_search` and code execution) and third-party integrations. Many ADK tools cannot be combined inside a single agent instance, so use this recommended pattern:

- Create specialized agents (for example, `SearchAgent`, `CodeAgent`).
- Wrap each specialized agent with an `AgentTool`.
- Orchestrate and delegate via a `RootAgent`.

This preserves clear responsibilities and avoids ADK tool conflicts.

Architecture example:

```bash
RootAgent
├─ AgentTool → AIDevSearchAgent → google_search
├─ AgentTool → CodeAgent → BuiltInCodeExecutor
├─ AgentTool → CodeExplainAgent
├─ AgentTool → HuggingFaceAgent → HF MCP → HF Hub
└─ AgentTool → GitHubAgent → GitHub MCP → GitHub
```

Notes:

- APIs belong to agents.
- Agents belong to the `RootAgent`.
- The `RootAgent` should not call APIs directly.

The application implements a state-aware, multi-agent architecture powered by Gemini models and Google ADK Web.

### Fine-tuning agent instructions

For reliable behavior, apply careful instruction engineering and strict behavioral controls.

In [ ]:
%%writefile app_01/agent.py
import os
import re
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor

# ======================================================
# Environment variables
# ======================================================
HF_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
# ======================================================
# Helper functions
# ======================================================
def extract_headlines(text: str):
    return re.findall(r"\d+\.\s*(.+)", text)


def extract_repo_candidate(text: str):
    """
    STRICT GitHub repo extraction.
    Returns owner/repo or None.
    """
    match = re.search(r"(?:github\.com/)?([\w\-]+/[\w\-]+)", text)
    return match.group(1) if match else None


def extract_hf_candidate(text: str):
    """
    STRICT Hugging Face ID extraction: org/name
    """
    match = re.search(r"\b([\w\-]+/[\w\-]+)\b", text)
    return match.group(1) if match else None


# ======================================================
# AI Developer News Agent
# ======================================================
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="AI developer news analyst with structured output.",
    instruction="""
You are an AI News Analyst for developers.

Rules:
- ONLY AI-related developer news
  If asked anything else, respond: "I can only provide recent AI-related developer news."
- ALWAYS use google_search
- DEFAULT to 3 articles if no number specified
- NEVER ask follow-up questions

Required output:

Using google_search, here are the top headlines:

---
[NUMBER]. HEADLINE

Summary:
1–2 sentence technical summary

Tech stack:
- frameworks / languages / infra OR Not mentioned

License:
- Open-source | Proprietary | Mixed | Not mentioned

GitHub repository:
- owner/repo if mentioned
- Otherwise: Not referenced

Hugging Face:
- Model / Dataset / Space if mentioned
- Otherwise: Not mentioned

Who should care:
- ML Engineer / Backend / MLOps / Data Scientist
---

End with:
"Which headline would you like to explore in more detail?"
""",
    tools=[google_search],
)

# ======================================================
# Python Execution Agent
# ======================================================
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Safe Python execution agent.",
    instruction="""
Execute Python safely.

Rules:
- No filesystem access
- No network calls
- No infinite loops
- Return result or error only
""",
    code_executor=BuiltInCodeExecutor(),
)

# ======================================================
# Python Explanation Agent
# ======================================================
code_explain_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeExplainAgent",
    description="Explains Python code safely.",
    instruction="""
Explain Python code step by step.
Do NOT execute or modify code.
""",
)

# ======================================================
# Hugging Face Canonical Reference Agent
# ======================================================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

hf_agent = Agent(
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    description="Returns canonical Hugging Face URLs for exact IDs.",
    instruction="""
Rules:
- ONLY accept exact Hugging Face IDs (org/name)
- NEVER guess or infer
- If not found, respond exactly:
"No Hugging Face resource found for the provided identifier."

Valid output ONLY:

Hugging Face URL:
https://huggingface.co/<exact_id>
""",
    tools=(
        [
            McpToolset(
                connection_params=StdioConnectionParams(
                    server_params=StdioServerParameters(
                        command="npx",
                        args=["-y", "@llmindset/hf-mcp-server"],
                        env={"HF_TOKEN": HF_TOKEN},
                    ),
                    timeout=30,
                ),
            )
        ]
        if HF_TOKEN
        else []
    ),
)

# ======================================================
# GitHub MCP Agent (STRICT, DATA-ONLY)
# ======================================================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

git_agent = Agent(
    model="gemini-2.5-flash",
    name="github_agent",
    description="GitHub MCP repository inspector.",
    instruction="""
Input will be EXACT owner/repo.

MANDATORY:
- Call GitHub MCP
- No guessing
- No prose
- If MCP fails, say exactly:
  "No GitHub repository found."

Output format ONLY:

Repository: owner/repo
Stars: <number>
Forks: <number>
Open issues: <number>
Open PRs: <number>
Recent activity:
- Commits (30d): <number>
- Last commit date: <date>
Overall activity level: High | Medium | Low
""",
    tools=[
        McpToolset(
            connection_params=StreamableHTTPServerParams(
                url="https://api.githubcopilot.com/mcp/",
                headers={
                    "Authorization": f"Bearer {GITHUB_TOKEN}",
                    "X-MCP-Toolsets": "all",
                    "X-MCP-Readonly": "true",
                },
            )
        )
    ] if GITHUB_TOKEN else [],
)

# ======================================================
# Root Routing Agent
# ======================================================
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Strict routing agent.",
    instruction="""
Routing rules:
- AI news → AIDevSearchAgent
- Python execution → CodeAgent
- Python explanation → CodeExplainAgent
- Hugging Face links → hugging_face_agent
- GitHub repos → github_agent

Rules:
- ALWAYS delegate
- NEVER answer directly
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
        AgentTool(agent=code_explain_agent),
        AgentTool(agent=hf_agent),
        AgentTool(agent=git_agent),
    ],
)

# ======================================================
# Main Input Handler
# ======================================================
def handle_user_input(user_input: str, session_state: dict):
    if session_state is None:
        session_state = {"headlines": []}

    if user_input.lower() in {"exit", "quit"}:
        return "Goodbye!", {"headlines": []}

    # Python execution
    if user_input.lower().startswith("execute python code:"):
        code = user_input[len("execute python code:"):].strip()
        return coding_agent.run(code), session_state

    # Python explanation
    if user_input.lower().startswith("explain this python code:"):
        code = user_input[len("explain this python code:"):].strip()
        return code_explain_agent.run(code), session_state

# 🔴 FORCE GitHub routing FIRST
    repo = extract_repo_candidate(user_input)
    if repo and "/" in repo:
        return git_agent.run(repo), session_state

    # Python execution
    #if user_input.lower().startswith("execute python code:"):
        #code = user_input.split(":", 1)[1]
        #return coding_agent.run(code), session_state


# ️⃣ Explicit Hugging Face ID → MCP
    hf_id = extract_hf_candidate(user_input)
    if hf_id and "/" in hf_id:
        hf_result = hf_agent.run(hf_id)
        if hf_result:
            return hf_result, session_state

 # Default routing
    response = root_agent.run(user_input)
    session_state["headlines"] = extract_headlines(response)
    return response, session_state

